In [3]:
from sys import path
path.append('\\Program Files\\Microsoft.NET\\ADOMD.NET\\160')

In [4]:
from sys import path
for x in path:
    print(x, sep='\n')

c:\Users\Cory.Cundy\AppData\Local\Programs\Python\Python311\python311.zip
c:\Users\Cory.Cundy\AppData\Local\Programs\Python\Python311\Lib
c:\Users\Cory.Cundy\AppData\Local\Programs\Python\Python311\DLLs

C:\Users\Cory.Cundy\AppData\Roaming\Python\Python311\site-packages
C:\Users\Cory.Cundy\AppData\Roaming\Python\Python311\site-packages\win32
C:\Users\Cory.Cundy\AppData\Roaming\Python\Python311\site-packages\win32\lib
C:\Users\Cory.Cundy\AppData\Roaming\Python\Python311\site-packages\Pythonwin
c:\Users\Cory.Cundy\AppData\Local\Programs\Python\Python311
c:\Users\Cory.Cundy\AppData\Local\Programs\Python\Python311\Lib\site-packages
C:\Windows\Microsoft.NET\Framework64\v4.0.30319\
\Program Files\Microsoft.NET\ADOMD.NET\160


In [7]:
import json
import pandas as pd
import adomd


json_path = "Sales & Returns Sample v201912.json"

with open(json_path, "r", encoding="utf-8-sig") as f:
    data = json.load(f)

# Flatten metrics into top-level columns
def flatten_event(event):
    flat = event.copy()
    metrics = flat.pop("metrics", {})
    flat.update(metrics)
    return flat

flat_events = [flatten_event(e) for e in data.get("events", [])]


# Load the 'events' list into a DataFrame
#df = pd.DataFrame(data.get("events", []))

df = pd.DataFrame(flat_events)

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 20)    
#print(df.head())

filtered_df = df[df['name'] == 'Execute DAX Query']
filtered_df


# Connection string to your Power BI semantic model (update as needed)
connection_string = "Provider=MSOLAP;Data Source=localhost:52438;Initial Catalog=eb3a32e1-bcb4-4163-902d-581f3f81346b"

results_list = []

# Loop through the DataFrame and execute each DAX query
with adomd.Pyadomd(connection_string) as conn:
    for idx, row in filtered_df.iterrows():
        query_id = row['id']
        query_text = row['QueryText']
        print(f"Executing Query ID: {query_id}")
        #print(f"QueryText:\n{query_text}\n")
        try:
            with conn.cursor().execute(query_text) as cur:
                results = cur.fetchall()
                print(f"Results for {query_id}: {results}\n")
        except Exception as e:
            print(f"Error executing query {query_id}: {e}\n")


NameError: name 'AdomdConnection' is not defined

In [7]:
import json
import pandas as pd
from pyadomd import Pyadomd


path.append('\\Program Files\\Microsoft.NET\\ADOMD.NET\\160')

json_path = "Sales & Returns Sample v201912.json"

with open(json_path, "r", encoding="utf-8-sig") as f:
    data = json.load(f)

# Flatten metrics into top-level columns
def flatten_event(event):
    flat = event.copy()
    metrics = flat.pop("metrics", {})
    flat.update(metrics)
    return flat

flat_events = [flatten_event(e) for e in data.get("events", [])]


# Load the 'events' list into a DataFrame
#df = pd.DataFrame(data.get("events", []))

df = pd.DataFrame(flat_events)

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 20)    
#print(df.head())

filtered_df = df[df['name'] == 'Execute DAX Query']
filtered_df


# Connection string to your Power BI semantic model (update as needed)
connection_string = "Provider=MSOLAP;Data Source=localhost:61954;Initial Catalog=eb3a32e1-bcb4-4163-902d-581f3f81346b"

results_list = []

# Loop through the DataFrame and execute each DAX query
with Pyadomd(connection_string) as conn:
    for idx, row in filtered_df.iterrows():
        query_id = row['id']
        query_text = row['QueryText']
        print(f"Executing Query ID: {query_id}")
        #print(f"QueryText:\n{query_text}\n")
        try:
            with conn.cursor().execute(query_text) as cur:
                results = cur.fetchall()
                print(f"Results for {query_id}: {results}\n")
        except Exception as e:
            print(f"Error executing query {query_id}: {e}\n")


NameError: name 'AdomdConnection' is not defined

In [50]:
import json
import pandas as pd
from pyadomd import Pyadomd
import hashlib

path.append('\\Program Files\\Microsoft.NET\\ADOMD.NET\\160')

json_path = "Sales & Returns Sample v201912.json"

with open(json_path, "r", encoding="utf-8-sig") as f:
    data = json.load(f)

# Flatten metrics into top-level columns
def flatten_event(event):
    flat = event.copy()
    metrics = flat.pop("metrics", {})
    flat.update(metrics)
    return flat

flat_events = [flatten_event(e) for e in data.get("events", [])]


# Load the 'events' list into a DataFrame
#df = pd.DataFrame(data.get("events", []))

df = pd.DataFrame(flat_events)

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 20)    
#print(df.head())

filtered_df = df[df['name'] == 'Execute DAX Query'].copy()
filtered_df



#print(filtered_df.nunique())


# Connection string to your Power BI semantic model (update as needed)
connection_string = "Provider=MSOLAP;Data Source=localhost:52438;Initial Catalog=eb3a32e1-bcb4-4163-902d-581f3f81346b"

results_list = []

# Loop through the DataFrame and execute each DAX query
with Pyadomd(connection_string) as conn:
    for idx, row in filtered_df.iterrows():
        query_id = row['id']
        query_text = row['QueryText']
        row_count = row['RowCount']
        print(f"Executing Query ID: {query_id}")
        # print(f"QueryText 3:\n{query_text}\n")
        try:
             with conn.cursor().execute(query_text) as cur:
                result_set_index = 0
                while True:
                    columns = [col[0] for col in cur.description]
                    result_rows = cur.fetchall()
                    print(f"Result set {result_set_index} has {len(result_rows)} rows")
                    if result_rows:
                        # Create a DataFrame for this query's results
                        result_df = pd.DataFrame(result_rows, columns=columns)
                        # Add a hash column for each row
                        def row_hash(row):
                            # Concatenate all values as string and hash
                            row_str = '|'.join(str(val) for val in row.values)
                            return hashlib.sha256(row_str.encode('utf-8')).hexdigest()
                        if not result_df.empty:
                            result_df['row_hash'] = result_df.apply(row_hash, axis=1)
                            print(f"query: {query_id} rows returned: {len(result_df)} rows: {row_count}")

                            # Sort by row_hash
                            result_df = result_df.sort_values('row_hash').reset_index(drop=True)
                            # Concatenate all row_hash values and hash them
                            all_hashes = ''.join(result_df['row_hash'].tolist())
                            combined_hash = hashlib.sha256(all_hashes.encode('utf-8')).hexdigest()
                            # Store in filtered_df (for the first result set only, or adjust as needed)
                            if result_set_index == 0:
                                filtered_df.loc[idx, 'combined_row_hash'] = combined_hash
                            print(f"query: {query_id} result set {result_set_index} rows: {len(result_df)} combined hash: {combined_hash}")
                        else:
                            print(f"query: {query_id} result set {result_set_index} is empty")

                        result_set_index += 1
                        if not cur.NextResult():
                            break

        except Exception as e:
            print(f"Error executing query {query_id}: {e}\n")

# Combine all results into a DataFrame
#results_df = pd.DataFrame(results_list)
filtered_df

Executing Query ID: 82bd947a-c0cb-48f5-87aa-2db65515b25e
Result set 0 has 1 rows
query: 82bd947a-c0cb-48f5-87aa-2db65515b25e rows returned: 1 rows: 1.0
query: 82bd947a-c0cb-48f5-87aa-2db65515b25e result set 0 rows: 1 combined hash: dfaa90b9f1d0497f2836e7a73b6e332125ba378ff023382491970708da76b0d5
Error executing query 82bd947a-c0cb-48f5-87aa-2db65515b25e: 'Cursor' object has no attribute 'NextResult'

Executing Query ID: baa3553b-c0fa-4f0e-87d6-98ea260fecba
Result set 0 has 1 rows
query: baa3553b-c0fa-4f0e-87d6-98ea260fecba rows returned: 1 rows: 34.0
query: baa3553b-c0fa-4f0e-87d6-98ea260fecba result set 0 rows: 1 combined hash: dfaa90b9f1d0497f2836e7a73b6e332125ba378ff023382491970708da76b0d5
Error executing query baa3553b-c0fa-4f0e-87d6-98ea260fecba: 'Cursor' object has no attribute 'NextResult'

Executing Query ID: 09fa85a8-b263-484c-a590-1fb11a89288a
Result set 0 has 1 rows
query: 09fa85a8-b263-484c-a590-1fb11a89288a rows returned: 1 rows: 1.0
query: 09fa85a8-b263-484c-a590-1fb11a89

,name,component,start,id,sourceLabel,end,status,visualTitle,visualId,visualType,initialLoad,parentId,QueryText,RowCount,combined_row_hash
37,Execute DAX Query,DSE,2025-05-28T16:12:14.236Z,82bd947a-c0cb-48f5-87aa-2db65515b25e,NaN,2025-05-28T16:12:14.249Z,NaN,NaN,NaN,NaN,NaN,14e4cc77-0fae-44fa-90db-9708a86ef10f,"DEFINE VAR __DS0FilterTable = \r\n\tTREATAS({""...",1.0,dfaa90b9f1d0497f2836e7a73b6e332125ba378ff02338...
69,Execute DAX Query,DSE,2025-05-28T16:12:14.236Z,baa3553b-c0fa-4f0e-87d6-98ea260fecba,NaN,2025-05-28T16:12:14.249Z,NaN,NaN,NaN,NaN,NaN,44f14ad4-8cdf-4d6b-9574-d8c4cbdc5e60,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tTRE...,34.0,dfaa90b9f1d0497f2836e7a73b6e332125ba378ff02338...
98,Execute DAX Query,DSE,2025-05-28T16:12:17.026Z,09fa85a8-b263-484c-a590-1fb11a89288a,NaN,2025-05-28T16:12:17.034Z,NaN,NaN,NaN,NaN,NaN,e2c0fd11-2c16-42a2-ba70-68c8bfe0eb41,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tTRE...,1.0,a587cdfbde2b7d9e4235afdeadd82d6e0f1cae0cd77a70...
111,Execute DAX Query,DSE,2025-05-28T16:12:17.051Z,206771da-ca77-40ad-a75b-280d0d7a62eb,NaN,2025-05-28T16:12:17.057Z,NaN,NaN,NaN,NaN,NaN,ba8a86ad-88b7-4d0e-a6cc-e76c8ae7479f,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tTRE...,1.0,ed699f7e2a5a74c67bc00eec9c0a7f2e4f272ceffcefce...
128,Execute DAX Query,DSE,2025-05-28T16:12:17.051Z,9f6684fc-b91b-4b9a-9650-4edb74369753,NaN,2025-05-28T16:12:17.061Z,NaN,NaN,NaN,NaN,NaN,be22ce78-faee-4c00-bf3f-fa40afb7547c,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tTRE...,1.0,ec54632ffc14ba0fbb9968a221888c367dc6bbcf3dd6d6...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
709,Execute DAX Query,DSE,2025-05-28T16:12:28.714Z,de5a7a49-e8bd-4659-8f45-a32f6eb2a076,NaN,2025-05-28T16:12:28.730Z,NaN,NaN,NaN,NaN,NaN,3506cac7-3be4-4c57-96cf-77cebe329334,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,dfaa90b9f1d0497f2836e7a73b6e332125ba378ff02338...
724,Execute DAX Query,DSE,2025-05-28T16:12:28.714Z,de5a7a49-e8bd-4659-8f45-a32f6eb2a076,NaN,2025-05-28T16:12:28.730Z,NaN,NaN,NaN,NaN,NaN,3506cac7-3be4-4c57-96cf-77cebe329334,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,dfaa90b9f1d0497f2836e7a73b6e332125ba378ff02338...
739,Execute DAX Query,DSE,2025-05-28T16:12:28.714Z,de5a7a49-e8bd-4659-8f45-a32f6eb2a076,NaN,2025-05-28T16:12:28.730Z,NaN,NaN,NaN,NaN,NaN,3506cac7-3be4-4c57-96cf-77cebe329334,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,dfaa90b9f1d0497f2836e7a73b6e332125ba378ff02338...
760,Execute DAX Query,DSE,2025-05-28T16:12:28.734Z,20e49a5f-3c16-4761-a702-2e3b936b60b0,NaN,2025-05-28T16:12:28.740Z,NaN,NaN,NaN,NaN,NaN,ddaf755a-216b-48e8-b284-cd51c8141f83,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,405c0c53ffc90007579bdaccf8bccaea39691b89065c1f...


In [4]:
import json
import pandas as pd
import adomd


json_path = "Sales & Returns Sample v201912.json"

with open(json_path, "r", encoding="utf-8-sig") as f:
    data = json.load(f)

# Flatten metrics into top-level columns
def flatten_event(event):
    flat = event.copy()
    metrics = flat.pop("metrics", {})
    flat.update(metrics)
    return flat

flat_events = [flatten_event(e) for e in data.get("events", [])]


# Load the 'events' list into a DataFrame
#df = pd.DataFrame(data.get("events", []))

df = pd.DataFrame(flat_events)

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 20)    
#print(df.head())

filtered_df = df[df['name'] == 'Execute DAX Query']
filtered_df


# Connection string to your Power BI semantic model (update as needed)
connection_string = "Provider=MSOLAP;Data Source=localhost:52438;Initial Catalog=eb3a32e1-bcb4-4163-902d-581f3f81346b"

results_list = []

# Loop through the DataFrame and execute each DAX query
with adomd.Pyadomd(connection_string) as conn:
    for idx, row in filtered_df.iterrows():
        query_id = row['id']
        query_text = row['QueryText']
        print(f"Executing Query ID: {query_id}")
        #print(f"QueryText:\n{query_text}\n")
        try:
            with conn.cursor().execute(query_text) as cur:
                results = cur.fetchall()
                print(f"Results for {query_id}: {results}\n")

                t = cur.nextresult()
                print(f"Next result: {t}")
        except Exception as e:
            print(f"Error executing query {query_id}: {e}\n")


NameError: name 'AdomdConnection' is not defined